In [4]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone

# ==========
# INPUTS
# ==========
HELIUS_API_KEY = "ced71d94-9f24-447f-aa30-78f36a8d6f6b"
MINT = "B8RHrVBxSjBGKqAbn1tXo6CWjvt5jFkkqjbCZtuDpump"
ORIGINAL_SUPPLY = 1_000_000_000

RPC_URL = f"https://mainnet.helius-rpc.com/?api-key={HELIUS_API_KEY}"
PARSE_TX_URL = f"https://api-mainnet.helius-rpc.com/v0/transactions?api-key={HELIUS_API_KEY}"

# ==========
# HELPERS
# ==========
def rpc_call(method, params):
    payload = {
        "jsonrpc": "2.0",
        "id": "1",
        "method": method,
        "params": params
    }
    r = requests.post(RPC_URL, json=payload, timeout=30)
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        raise Exception(data["error"])
    return data["result"]

def get_token_supply(mint):
    result = rpc_call("getTokenSupply", [mint])
    value = result["value"]
    amount_raw = int(value["amount"])
    decimals = int(value["decimals"])
    ui_amount = amount_raw / (10 ** decimals)
    return {
        "mint": mint,
        "amount_raw": amount_raw,
        "decimals": decimals,
        "ui_amount": ui_amount,
    }

def get_signatures_for_address(address, before=None, limit=1000):
    params = [address, {"limit": limit}]
    if before:
        params[1]["before"] = before
    return rpc_call("getSignaturesForAddress", params)

def parse_transactions(signatures):
    if not signatures:
        return []
    r = requests.post(PARSE_TX_URL, json={"transactions": signatures}, timeout=60)
    r.raise_for_status()
    return r.json()

def chunk_list(items, n):
    for i in range(0, len(items), n):
        yield items[i:i+n]

def extract_burn_rows(parsed_txs, mint):
    rows = []

    for tx in parsed_txs:
        if not isinstance(tx, dict):
            continue

        tx_type = tx.get("type")
        if tx_type != "BURN":
            continue

        signature = tx.get("signature")
        timestamp = tx.get("timestamp")
        dt = datetime.fromtimestamp(timestamp, tz=timezone.utc) if timestamp else None
        description = tx.get("description")
        slot = tx.get("slot")
        source = tx.get("source")
        fee = tx.get("fee")

        amount_raw = None
        amount_ui = None
        decimals = None
        from_user = None
        from_token = None

        # First try tokenTransfers
        for tr in tx.get("tokenTransfers", []) or []:
            if tr.get("mint") != mint:
                continue

            raw = tr.get("rawTokenAmount")
            ui = tr.get("tokenAmount")
            dec = tr.get("decimals")

            if raw is not None:
                try:
                    amount_raw = int(raw)
                except:
                    pass

            if ui is not None:
                try:
                    amount_ui = float(ui)
                except:
                    pass

            if dec is not None:
                try:
                    decimals = int(dec)
                except:
                    pass

            from_user = tr.get("fromUserAccount")
            from_token = tr.get("fromTokenAccount")
            break

        # Fallback: infer from events if tokenTransfers doesn't expose it
        if amount_raw is None:
            events = tx.get("events", {}) or {}
            if isinstance(events, dict):
                for _, v in events.items():
                    if isinstance(v, dict) and v.get("mint") == mint:
                        if v.get("amount") is not None:
                            try:
                                amount_raw = int(v["amount"])
                            except:
                                pass
                        if v.get("tokenAmount") is not None:
                            try:
                                amount_ui = float(v["tokenAmount"])
                            except:
                                pass
                        break

        # Derive ui from raw if needed
        if amount_ui is None and amount_raw is not None:
            if decimals is None:
                # DUSD appears to be 6 decimals from getTokenSupply
                decimals = 6
            amount_ui = amount_raw / (10 ** decimals)

        rows.append({
            "signature": signature,
            "timestamp": timestamp,
            "datetime_utc": dt,
            "type": tx_type,
            "slot": slot,
            "source": source,
            "fee_lamports": fee,
            "amount_raw": amount_raw,
            "amount_ui": amount_ui,
            "decimals": decimals,
            "from_user": from_user,
            "from_token_account": from_token,
            "description": description,
        })

    return rows

def backfill_burns_via_signatures(mint, max_signature_pages=10, sig_page_size=1000, parse_batch_size=100, sleep_s=0.35):
    """
    1) pull signatures for mint
    2) parse them in batches
    3) keep BURN txs
    """
    all_signatures = []
    before = None

    for page in range(max_signature_pages):
        sig_infos = get_signatures_for_address(mint, before=before, limit=sig_page_size)
        if not sig_infos:
            print(f"No more signatures on page {page+1}")
            break

        page_sigs = [x["signature"] for x in sig_infos if "signature" in x]
        all_signatures.extend(page_sigs)
        print(f"Signature page {page+1}: {len(page_sigs)} signatures")

        before = sig_infos[-1]["signature"]

        # gentle pacing
        time.sleep(sleep_s)

    print(f"Total signatures fetched: {len(all_signatures)}")

    burn_rows = []
    for i, batch in enumerate(chunk_list(all_signatures, parse_batch_size), start=1):
        parsed = parse_transactions(batch)
        rows = extract_burn_rows(parsed, mint)
        burn_rows.extend(rows)
        print(f"Parsed batch {i}: {len(batch)} sigs -> {len(rows)} burn rows")
        time.sleep(sleep_s)

    df = pd.DataFrame(burn_rows)
    if not df.empty:
        df = df.sort_values("timestamp", ascending=False).reset_index(drop=True)
    return df

# ==========
# RUN
# ==========
supply = get_token_supply(MINT)
print("SUPPLY:", supply)

current_supply = supply["ui_amount"]
total_burned = ORIGINAL_SUPPLY - current_supply

print(f"Current supply: {current_supply:,.6f}")
print(f"Total burned:   {total_burned:,.6f}")

burn_df = backfill_burns_via_signatures(
    MINT,
    max_signature_pages=5,   # increase later
    sig_page_size=1000,
    parse_batch_size=100,
    sleep_s=0.4
)

print("\nBURN HEAD")
print(burn_df.head(20))

if not burn_df.empty:
    now_ts = datetime.now(timezone.utc).timestamp()
    burned_24h = burn_df.loc[burn_df["timestamp"] >= now_ts - 86400, "amount_ui"].fillna(0).sum()
    burned_7d = burn_df.loc[burn_df["timestamp"] >= now_ts - 7*86400, "amount_ui"].fillna(0).sum()
    burned_30d = burn_df.loc[burn_df["timestamp"] >= now_ts - 30*86400, "amount_ui"].fillna(0).sum()

    print(f"\nBurned 24h: {burned_24h:,.6f}")
    print(f"Burned 7d:  {burned_7d:,.6f}")
    print(f"Burned 30d: {burned_30d:,.6f}")

burn_df.to_csv("dusd_burn_log.csv", index=False)
print("\nSaved dusd_burn_log.csv")

SUPPLY: {'mint': 'B8RHrVBxSjBGKqAbn1tXo6CWjvt5jFkkqjbCZtuDpump', 'amount_raw': 677675569853228, 'decimals': 6, 'ui_amount': 677675569.853228}
Current supply: 677,675,569.853228
Total burned:   322,324,430.146772
Signature page 1: 1000 signatures
Signature page 2: 1000 signatures
Signature page 3: 1000 signatures
Signature page 4: 1000 signatures
Signature page 5: 1000 signatures
Total signatures fetched: 5000
Parsed batch 1: 100 sigs -> 1 burn rows
Parsed batch 2: 100 sigs -> 0 burn rows
Parsed batch 3: 100 sigs -> 0 burn rows
Parsed batch 4: 100 sigs -> 0 burn rows
Parsed batch 5: 100 sigs -> 1 burn rows
Parsed batch 6: 100 sigs -> 0 burn rows
Parsed batch 7: 100 sigs -> 0 burn rows
Parsed batch 8: 100 sigs -> 1 burn rows
Parsed batch 9: 100 sigs -> 4 burn rows
Parsed batch 10: 100 sigs -> 0 burn rows
Parsed batch 11: 100 sigs -> 0 burn rows
Parsed batch 12: 100 sigs -> 2 burn rows
Parsed batch 13: 100 sigs -> 2 burn rows
Parsed batch 14: 100 sigs -> 1 burn rows
Parsed batch 15: 100 s

In [6]:
import requests
import pandas as pd

HELIUS_API_KEY = "ced71d94-9f24-447f-aa30-78f36a8d6f6b"
MINT = "B8RHrVBxSjBGKqAbn1tXo6CWjvt5jFkkqjbCZtuDpump"

RPC_URL = f"https://mainnet.helius-rpc.com/?api-key={HELIUS_API_KEY}"

TOKEN_PROGRAM_ID = "TokenkegQfeZyiNwAJbNbGKPFXCWuBvf9Ss623VQ5DA"

def rpc_call(method, params):
    payload = {
        "jsonrpc": "2.0",
        "id": "1",
        "method": method,
        "params": params
    }
    r = requests.post(RPC_URL, json=payload, timeout=60)
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        raise Exception(data["error"])
    return data["result"]

def get_all_token_accounts_for_mint_v2(mint, limit=1000):
    """
    Pull all token accounts for a mint using getProgramAccountsV2 pagination.
    """
    all_accounts = []
    pagination_key = None

    while True:
        params_obj = {
            "encoding": "jsonParsed",
            "limit": limit,
            "filters": [
                {"dataSize": 165},  # SPL token account size
                {
                    "memcmp": {
                        "offset": 0,
                        "bytes": mint
                    }
                }
            ]
        }

        if pagination_key:
            params_obj["paginationKey"] = pagination_key

        result = rpc_call("getProgramAccountsV2", [TOKEN_PROGRAM_ID, params_obj])

        accounts = result.get("accounts", [])
        pagination_key = result.get("paginationKey")

        all_accounts.extend(accounts)
        print(f"Fetched {len(accounts)} accounts, total so far: {len(all_accounts)}")

        if not pagination_key:
            break

    return all_accounts

def count_holders(accounts):
    """
    Count unique non-zero owners.
    """
    owners = set()
    rows = []

    for acct in accounts:
        data = acct.get("account", {}).get("data", {}).get("parsed", {}).get("info", {})
        owner = data.get("owner")
        token_amount = data.get("tokenAmount", {})
        raw_amount = token_amount.get("amount", "0")
        ui_amount = token_amount.get("uiAmount", 0)

        if owner is None:
            continue

        # ignore zero-balance token accounts
        if raw_amount == "0":
            continue

        owners.add(owner)
        rows.append({
            "owner": owner,
            "raw_amount": raw_amount,
            "ui_amount": ui_amount
        })

    df = pd.DataFrame(rows)
    return len(owners), df

accounts = get_all_token_accounts_for_mint_v2(MINT, limit=1000)
holder_count, holder_df = count_holders(accounts)

print(f"\nTotal non-zero unique holders: {holder_count}")
print(holder_df.head(10))

Fetched 1000 accounts, total so far: 1000
Fetched 1000 accounts, total so far: 2000
Fetched 1000 accounts, total so far: 3000
Fetched 1000 accounts, total so far: 4000
Fetched 1000 accounts, total so far: 5000
Fetched 837 accounts, total so far: 5837
Fetched 0 accounts, total so far: 5837

Total non-zero unique holders: 1615
                                          owner    raw_amount      ui_amount
0  E9vqwa21mMHbbRdSzENMhX3b9jvQwmtpVVjkXcU48jfr     541386523     541.386523
1  C8nGF6HPgpYhoqvVPmgr6ChBZusXRbJ4UToLbH8Xphoy  346127263530  346127.263530
2  4UQN45DxGqwrye6o2uFd7bUAjs9jxfRjLnYrz2pB97pN             1       0.000001
3  HU2BsELeWFVaCWAXDZrqrMw8VbkSbHm7JvHBBtwD3Nfe  124248746431  124248.746431
4  AP57Wa3H7y3MJ3dQ8F7L9Xjjv3eBBsWRoJpzGiZouPPp  193364685161  193364.685161
5   UTqEuKtdkDgHg6S5viZLwr9boofXFQZgofphf58bE7V             1       0.000001
6  2a2NqsUWFR8eFaXNkhhuFSpVkTbctGao9ZZvbkLGcrEm    1611531864    1611.531864
7  GwV41YwsQNDLsNQKToNMH1wVnxzFrJWvRF2f7NYzr2Kb          

In [7]:
import requests
import pandas as pd
from datetime import datetime, timezone

MINT = "B8RHrVBxSjBGKqAbn1tXo6CWjvt5jFkkqjbCZtuDpump"

def fetch_dexscreener_pairs(chain_id, token_address):
    url = f"https://api.dexscreener.com/token-pairs/v1/{chain_id}/{token_address}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def choose_best_pair(pairs):
    """
    Prefer highest liquidity USD.
    """
    if not pairs:
        return None

    def liq_usd(p):
        liq = p.get("liquidity") or {}
        return float(liq.get("usd") or 0)

    pairs_sorted = sorted(pairs, key=liq_usd, reverse=True)
    return pairs_sorted[0]

pairs = fetch_dexscreener_pairs("solana", MINT)
print(f"Pairs returned: {len(pairs)}")

best = choose_best_pair(pairs)
print("\nBEST PAIR RAW:")
print(best)

if best:
    out = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "chain_id": best.get("chainId"),
        "dex_id": best.get("dexId"),
        "pair_address": best.get("pairAddress"),
        "pair_url": best.get("url"),
        "base_symbol": (best.get("baseToken") or {}).get("symbol"),
        "quote_symbol": (best.get("quoteToken") or {}).get("symbol"),
        "price_usd": float(best.get("priceUsd") or 0),
        "liquidity_usd": float((best.get("liquidity") or {}).get("usd") or 0),
        "volume_24h": float((best.get("volume") or {}).get("h24") or 0),
        "buys_24h": int(((best.get("txns") or {}).get("h24") or {}).get("buys") or 0),
        "sells_24h": int(((best.get("txns") or {}).get("h24") or {}).get("sells") or 0),
        "fdv": float(best.get("fdv") or 0),
        "market_cap": float(best.get("marketCap") or 0),
        "price_change_24h_pct": float((best.get("priceChange") or {}).get("h24") or 0),
        "pair_created_at": best.get("pairCreatedAt"),
    }

    df = pd.DataFrame([out])
    print("\nPARSED SNAPSHOT:")
    print(df.T)

    df.to_csv("dusd_dexscreener_test.csv", index=False)
    print("\nSaved dusd_dexscreener_test.csv")
else:
    print("No pair found.")

Pairs returned: 5

BEST PAIR RAW:
{'chainId': 'solana', 'dexId': 'pumpswap', 'url': 'https://dexscreener.com/solana/2z2ubdxaafishs6xrz2atv2ro7jl8hfduckxttfhjhdw', 'pairAddress': '2z2uBDXaAFishs6XRz2atV2Ro7jL8hfdUcKxTTfhjHDw', 'baseToken': {'address': 'B8RHrVBxSjBGKqAbn1tXo6CWjvt5jFkkqjbCZtuDpump', 'name': 'Deflationary USD', 'symbol': 'DUSD'}, 'quoteToken': {'address': 'So11111111111111111111111111111111111111112', 'name': 'Wrapped SOL', 'symbol': 'SOL'}, 'priceNative': '0.000002111', 'priceUsd': '0.0001984', 'txns': {'m5': {'buys': 1, 'sells': 0}, 'h1': {'buys': 7, 'sells': 2}, 'h6': {'buys': 19, 'sells': 6}, 'h24': {'buys': 91, 'sells': 43}}, 'volume': {'h24': 4060.75, 'h6': 837.79, 'h1': 202.39, 'm5': 10.17}, 'priceChange': {'m5': 0.28, 'h1': 0.58, 'h6': -3.09, 'h24': -16.01}, 'liquidity': {'usd': 46860.85, 'base': 118177156, 'quote': 249.1976}, 'fdv': 134465, 'marketCap': 134465, 'pairCreatedAt': 1757092196000, 'info': {'imageUrl': 'https://cdn.dexscreener.com/cms/images/mFdXRN9HxO